In [10]:
import os
import requests
from urllib.parse import quote
from pathlib import Path


In [11]:
Path(os.path.abspath(os.path.join(os.path.dirname('__file__')))).parent

PosixPath('/Users/nikitakuznecov/Desktop/petprojects/market_narratives_engine')

In [12]:

REPO_ID = "rjac/all-the-news-2-1-Component-one"
REVISION = "main"

# Change this if needed:
# - "data" (most common)
# - "date" (if your target folder is named exactly like this)
START_PATH = "data"

# Local root where the repo structure will be recreated
root_dir = Path(os.path.abspath(os.path.join(os.path.dirname('__file__')))).parent
OUT_ROOT = os.path.join(root_dir, "data", "raw", REPO_ID.replace("/", "__"))

API_BASE = "https://huggingface.co/api/datasets"
FILE_BASE = "https://huggingface.co/datasets"

# Optional for private/gated repos:
HF_TOKEN = os.getenv("HF_TOKEN")  # leave empty for public repos

session = requests.Session()
if HF_TOKEN:
    session.headers.update({"Authorization": f"Bearer {HF_TOKEN}"})



In [13]:

def list_tree(repo_id: str, revision: str, subpath: str):
    """
    List direct children of `subpath` in dataset repo.
    """
    subpath = subpath.strip("/")
    if subpath:
        url = f"{API_BASE}/{repo_id}/tree/{revision}/{quote(subpath, safe='/')}"
    else:
        url = f"{API_BASE}/{repo_id}/tree/{revision}"
    r = session.get(url, timeout=60)
    r.raise_for_status()
    return r.json()


def walk_files(repo_id: str, revision: str, start_path: str):
    """
    DFS through directories, yielding file paths.
    """
    stack = [start_path.strip("/")]
    while stack:
        current = stack.pop()
        items = list_tree(repo_id, revision, current)

        for item in items:
            item_type = item.get("type")
            item_path = item.get("path")

            if not item_path:
                continue

            if item_type == "directory":
                stack.append(item_path)
            elif item_type == "file":
                yield item_path


def download_file(repo_id: str, revision: str, file_path: str, out_root: str):
    """
    Download one file from HF and save with original relative path.
    """
    url = f"{FILE_BASE}/{repo_id}/resolve/{revision}/{quote(file_path, safe='/')}"
    local_path = os.path.join(out_root, *file_path.split("/"))
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    # Skip already downloaded non-empty files
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        return "skip", local_path

    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return "ok", local_path


In [14]:

def main():
    os.makedirs(OUT_ROOT, exist_ok=True)

    files = list(walk_files(REPO_ID, REVISION, START_PATH))
    if not files:
        print(f"No files found under START_PATH='{START_PATH}'.")
        return

    print(f"Found {len(files)} files under '{START_PATH}'")
    ok = skip = fail = 0

    for i, fp in enumerate(files, 1):
        try:
            status, local = download_file(REPO_ID, REVISION, fp, OUT_ROOT)
            if status == "ok":
                ok += 1
            else:
                skip += 1
            print(f"[{i}/{len(files)}] {status.upper()}  {fp}")
        except Exception as e:
            fail += 1
            print(f"[{i}/{len(files)}] FAIL  {fp}  -> {e}")

    print("\nDone")
    print(f"Downloaded: {ok}")
    print(f"Skipped:    {skip}")
    print(f"Failed:     {fail}")
    print(f"Saved to:   {OUT_ROOT}")


if __name__ == "__main__":
    main()

Found 36 files under 'data'
[1/36] OK  data/train-00000-of-00036-44ab9c81d9134716.parquet
[2/36] OK  data/train-00001-of-00036-f36e9ee23e3bf9b5.parquet
[3/36] OK  data/train-00002-of-00036-c0ea38dcfe02535f.parquet
[4/36] OK  data/train-00003-of-00036-7caf126bb260acdc.parquet
[5/36] OK  data/train-00004-of-00036-9eb935cffca9f23b.parquet
[6/36] OK  data/train-00005-of-00036-0996ab3bf2ae5582.parquet
[7/36] OK  data/train-00006-of-00036-cc1025af919f2e1d.parquet
[8/36] OK  data/train-00007-of-00036-f4520d6332b95d0b.parquet
[9/36] OK  data/train-00008-of-00036-ab8ebf54718ca5e9.parquet
[10/36] OK  data/train-00009-of-00036-97d6ee7daf701bb8.parquet
[11/36] OK  data/train-00010-of-00036-05d6e7faa69f27d6.parquet
[12/36] OK  data/train-00011-of-00036-f081b2f0269c6441.parquet
[13/36] OK  data/train-00012-of-00036-dac3a0ea6ba05872.parquet
[14/36] OK  data/train-00013-of-00036-b2bb7a62558c08b8.parquet
[15/36] OK  data/train-00014-of-00036-26e1d691069e228a.parquet
[16/36] OK  data/train-00015-of-0003